In [1]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table


ModuleNotFoundError: No module named 'utils'

In [ ]:
df = pd.read_csv("../../data/dosm_job_demand.csv")

In [ ]:
# -------------------------
# 1. Create date column
# -------------------------
quarter_map = {
    1: "01-01",
    2: "04-01",
    3: "07-01",
    4: "10-01"
}

df["Quarter"] = df["Quarter"].astype(int)

df["date"] = pd.to_datetime(
    df["Year"].astype(str) + "-" + df["Quarter"].map(quarter_map)
)

In [ ]:
# -------------------------
# 2. Rename columns
# -------------------------
df = df.rename(columns={
    "Skills": "skill_level",
    "Economic Activity": "sector",
    "Sub-economic Activity": "subsector",
    "Jobs ('000)": "job_available",
    "Filled Jobs ('000)": "job_filled",
    "Vacancies ('000)": "job_vacancy",
    "Jobs Created ('000)": "job_created"
})

In [ ]:
# -------------------------
# 3. Convert numeric columns
# remove commas and multiply by 1000
# -------------------------
num_cols = [
    "job_available",
    "job_filled",
    "job_vacancy",
    "job_created"
]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float) * 1000
    )

In [ ]:
# -------------------------
# 4. Select final columns
# -------------------------
df_final = df[[
    "date",
    "skill_level",
    "sector",
    "subsector",
    "job_available",
    "job_filled",
    "job_vacancy",
    "job_created"
]]
df_final["sector"] = df_final["sector"].replace({
    "Mining & Quarrying": "Mining and Quarrying"
})
df_final.head()

In [ ]:
# -------------------------
# 5. Aggregate by skill_level and sector
# -------------------------
df_final = df_final.groupby(['date','skill_level', 'sector'])[[
    'job_available',
    'job_filled',
    'job_vacancy',
    'job_created'
]].sum().reset_index()

df_final.head(20)

In [ ]:
write_table(df_final, "sc_bronze", "dosm_jobdemand")